In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    break

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("Folder:", root)
    print("Files:", files[:10])

In [ ]:
import os

path = "/kaggle/input/notebooks/saicharan2003/mnist"

print(os.listdir(path))

In [ ]:
import os

print(os.listdir("/kaggle/input"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/competitions"):
    print("Folder:", root)
    print("Files:", files)

In [ ]:
import pandas as pd

train_path = "/kaggle/input/competitions/digit-recognizer/train.csv"
test_path = "/kaggle/input/competitions/digit-recognizer/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()

In [ ]:
import matplotlib.pyplot as plt

# First image
image = train.iloc[0, 1:].values.reshape(28, 28)
label = train.iloc[0, 0]

plt.imshow(image, cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))

for i in range(9):
    image = train.iloc[i, 1:].values.reshape(28, 28)
    label = train.iloc[i, 0]

    plt.subplot(3, 3, i + 1)
    plt.imshow(image, cmap="gray")
    plt.title(f"Label: {label}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
X = train.drop("label", axis=1)
y = train["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X = X / 255.0

print("Minimum pixel value:", X.min().min())
print("Maximum pixel value:", X.max().max())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model.fit(X_train, y_train)

print("Model training completed!")

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

print("Validation Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_val, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=range(10)
)

disp.plot()
plt.title("MNIST Confusion Matrix")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))

for i in range(9):
    image = X_val.iloc[i].values.reshape(28, 28)
    actual = y_val.iloc[i]
    predicted = y_pred[i]

    plt.subplot(3, 3, i + 1)
    plt.imshow(image, cmap="gray")
    plt.title(f"Actual: {actual} | Predicted: {predicted}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
wrong = y_val.values != y_pred

print("Total wrong predictions:", wrong.sum())
print("Total correct predictions:", (~wrong).sum())

In [ ]:
from collections import Counter

wrong_actual = y_val.values[wrong]
wrong_predicted = y_pred[wrong]

pairs = Counter(zip(wrong_actual, wrong_predicted))

print("Most common mistakes:")

for (actual, predicted), count in pairs.most_common(10):
    print(f"Actual {actual} → Predicted {predicted}: {count} times")

In [ ]:
import matplotlib.pyplot as plt

wrong_indices = [i for i, value in enumerate(wrong) if value]

plt.figure(figsize=(10, 10))

for j in range(9):
    i = wrong_indices[j]

    image = X_val.iloc[i].values.reshape(28, 28)
    actual = y_val.iloc[i]
    predicted = y_pred[i]

    plt.subplot(3, 3, j + 1)
    plt.imshow(image, cmap="gray")
    plt.title(f"Actual: {actual} | Predicted: {predicted}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
X_test = test / 255.0

print("Test data shape:", X_test.shape)
print("Minimum:", X_test.min().min())
print("Maximum:", X_test.max().max())

In [ ]:
test_predictions = model.predict(X_test)

print("Predictions created!")
print("Total predictions:", len(test_predictions))
print("First 20 predictions:", test_predictions[:20])

In [ ]:
submission = pd.DataFrame({
    "ImageId": range(1, len(test_predictions) + 1),
    "Label": test_predictions
})

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")
print("Shape:", submission.shape)
submission.head()

In [ ]:
import matplotlib.pyplot as plt

accuracy = accuracy * 100

plt.figure(figsize=(6, 4))
plt.bar(["Logistic Regression"], [accuracy])

plt.ylim(0, 100)
plt.ylabel("Accuracy (%)")
plt.title("MNIST Model Accuracy")

plt.text(0, accuracy + 2, f"{accuracy:.2f}%", ha="center")

plt.show()